In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 18 — Reference coding solutions

Three exercises. Read the lesson and workbook first. Learner functions intentionally raise `NotImplementedError` until you write them. Reference execution is not your assessment.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/shape_lab").is_dir())
if str(ROOT/"src") not in sys.path:sys.path.insert(0,str(ROOT/"src"))


## 18.C1 — `fifo_age`

Simulate single-server FIFO completion age for strictly increasing finite arrivals and nonnegative service times.

**Before coding:** work a small example on paper.

**Hint:** Completion i = max(arrival i, previous completion)+service i.

In [3]:
def fifo_age(arrivals, service_times):
    a=np.asarray(arrivals,float);s=np.asarray(service_times,float)
    if a.ndim!=1 or a.shape!=s.shape or not np.isfinite(a).all() or not np.isfinite(s).all() or np.any(np.diff(a)<=0) or np.any(s<0):raise ValueError("Ordered arrivals and nonnegative times required.")
    finish=-np.inf;out=[]
    for t,d in zip(a,s):
        finish=max(finish,t)+d;out.append(finish-t)
    return np.asarray(out)

In [4]:
assert np.allclose(fifo_age([0,1,2],[2,2,2]),[2,3,4])
assert np.allclose(fifo_age([0,1,2],[.2,.2,.2]),[.2,.2,.2])
print("18.C1: checks passed")

18.C1: checks passed


**Why this works and where it stops:** This simulation shows queueing; it is not measured processing time and does not model parallel workers or dropped frames.

## 18.C2 — `latency_quantiles`

Return p50 and p95 for a nonempty finite nonnegative latency vector using NumPy default linear quantile interpolation.

**Before coding:** work a small example on paper.

**Hint:** Keep the unit unchanged and declare the quantile convention.

In [5]:
def latency_quantiles(latencies):
    x=np.asarray(latencies,float)
    if x.ndim!=1 or not len(x) or not np.isfinite(x).all() or np.any(x<0):raise ValueError("Finite nonnegative nonempty vector required.")
    return {"p50":float(np.quantile(x,.5)),"p95":float(np.quantile(x,.95))}

In [6]:
q=latency_quantiles([1,2,3,4,5])
assert q['p50']==3 and np.isclose(q['p95'],4.8)
print("18.C2: checks passed")

18.C2: checks passed


**Why this works and where it stops:** A p95 from five values is a descriptive statistic with very little tail evidence. More decimal places do not create more observations.

## 18.C3 — `vector_rigid`

Vectorize a rigid transform over a valid finite Nx3 point table. Inputs are already validated for this exercise.

**Before coding:** work a small example on paper.

**Hint:** Broadcast the translation over rows.

In [7]:
def vector_rigid(points, rotation, translation):
    return np.asarray(points,float)@np.asarray(rotation,float).T+np.asarray(translation,float)

In [8]:
from shape_lab.stereo import reference_sample,point_cloud
from shape_lab.geometry import exp_so3
_,_,d,c=reference_sample();X,_=point_cloud(d,c,stride=50);R=exp_so3([.1,0,.2]);t=np.array([1.,2.,3.])
assert np.allclose(vector_rigid(X,R,t),np.array([R@x+t for x in X]))
print("18.C3: checks passed")

18.C3: checks passed


**Why this works and where it stops:** Equal results must be verified before comparing performance. Vectorization does not remove the need to preserve source IDs or timestamps.